In [1]:
import os
base_path = os.path.dirname(os.path.abspath(""))
base_path = base_path + "/headers/"
print(base_path)
header_path = base_path + 'header_data_treatment.ipynb'
app_name = 'merge_universities'
%run -i $header_path 

C:\Users\raphael.wargon\Documents\GitHub\sorting_univ\script/headers/
merge_universities
initalizing spark nlp session w gpu


# Loading data

In [2]:
inst =  spark.read.parquet('file:\\' + save_path + 'intermediary\\inst_fr.parquet')

In [4]:
budget =  spark.read.json('file:\\' + scanR_path + '\\fr-esr-operateurs-indicateurs-financiers.json')

In [6]:
budget.printSchema()

root
 |-- academie: string (nullable = true)
 |-- acquisitions_d_immobilisations: double (nullable = true)
 |-- anr_hors_investissements_d_avenir: double (nullable = true)
 |-- anr_investissements_d_avenir: double (nullable = true)
 |-- autres_ressources_propres: double (nullable = true)
 |-- autres_subventions: double (nullable = true)
 |-- besoin_en_fonds_de_roulement: double (nullable = true)
 |-- caf_acquisitions_d_immobilisations: double (nullable = true)
 |-- caf_produits_encaissables: double (nullable = true)
 |-- capacite_d_autofinancement: double (nullable = true)
 |-- charges_de_fonctionnement_decaissables: double (nullable = true)
 |-- charges_de_personnel: double (nullable = true)
 |-- charges_de_personnel_produits_encaissables: double (nullable = true)
 |-- charges_decaissables_produits_encaissables: double (nullable = true)
 |-- charges_externes: double (nullable = true)
 |-- charges_externes_produits_encaissables: double (nullable = true)
 |-- code_de_l_academie: string 

In [10]:
budget.select('etablissement','etablissement_id_paysage').distinct().count()

201

# Normalisation functions for the match

In [11]:
from nltk.corpus import stopwords
print(stopwords.fileids())
stopwords_dict = {}
for language in stopwords.fileids():
    stopwords_dict[language.capitalize()] = stopwords.words(language)
    
def rm_stopwords(x,y):
    try:
        list_x = x.split(' ')
        list_y = y
        return(' '.join([word for word in list_x if word not in list_y]))
    except:
        return(x)
udf_rm_stopwords = func.udf(rm_stopwords)

['albanian', 'arabic', 'azerbaijani', 'basque', 'belarusian', 'bengali', 'catalan', 'chinese', 'danish', 'dutch', 'english', 'finnish', 'french', 'german', 'greek', 'hebrew', 'hinglish', 'hungarian', 'indonesian', 'italian', 'kazakh', 'nepali', 'norwegian', 'portuguese', 'romanian', 'russian', 'slovene', 'spanish', 'swedish', 'tajik', 'tamil', 'turkish', 'uzbek']


In [12]:
stopwords_dict['French'] = stopwords_dict['French'] + ['l']

In [13]:
from nltk.stem.snowball import FrenchStemmer
from nltk.stem import PorterStemmer


In [14]:

def stemmer(x,y):
    try:
        if y == 'fr' or y == 'French' or y == 'Français':
            output = ' '.join([FrenchStemmer().stem(word) for word in x.split(' ')])
            return(output)
        else:
            output = ' '.join([PorterStemmer().stem(word) for word in x.split(' ')])
            return(output)
    except:
        return(x)
udf_stemmer = func.udf(stemmer)

In [15]:
import unidecode
def unidecode_debug(x):
    try:
        return unidecode.unidecode(x) if x is not None else None
       
    except:
        return(x)

udf_unidecode = func.udf(unidecode_debug, StringType())

# Merge

In this part, we use observation-level characteristics to merge institutions between the two databases.
In each step : 
* we match by the characteristics related to the step
* we remove duplicates in this characteristic.

We perform this merge on  full name of the institution, acronym of the institution.

## Merge by name

In this part, we need to exclude some matches manually.

In [16]:
inst.show()

+-----------+--------------------+----------+-----------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+----+--------------------+-------------------+-------------------+---------+---------+---------------+-----------+--------------+
|    inst_id|                name|      type|       city|        homepage_url|         topic_share|          main_topic|              parent|uni_pub|           parent_id|cnrs|                idex|total_idex_dotation|            type_fr|  secteur|acces_rce|date_first_idex|fusion_date|merged_inst_id|
+-----------+--------------------+----------+-----------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+----+--------------------+-------------------+-------------------+---------+---------+---------------+-----------+--------------+
| I193033237|Institut Superieu...| education|   Toulouse|https://www.isae-...|[{Engineering, 0....| 

In [48]:
fr_oa_normalized_names = (inst.withColumn('is_oa', func.lit(1))
                           #.filter(~func.col('inst_id').isin(all_matched_inst_oa))
                              .withColumn('stopwords_fr',func.array([func.lit(x) for x in stopwords_dict['French']]))
                              .withColumn('stopwords_en',func.array([func.lit(x) for x in stopwords_dict['English']]))

                    .filter(func.col('name').isNotNull())
                   .withColumn('name', func.regexp_replace(func.col('name'), r"\([A-z\s]*\)", ''))
                  .withColumn('name', func.regexp_replace(func.col('name'),  r"""[!\"#$%&'()*+,\-.\/:;<=>?@\[\\\]^_`{|}~]""", ' '))
                   .withColumn('name', func.regexp_replace(func.col('name'),  r"\s{2,}", ' ')) 
                  .withColumn('name', func.regexp_replace(func.col('name'), r"\s$", ''))
                        
                              .withColumn('name', udf_rm_stopwords(func.lower(func.col('name')),func.col('stopwords_fr')))
                              .withColumn('name', udf_rm_stopwords(func.lower(func.col('name')), func.col('stopwords_en')))
                   .filter(func.col('name').isNotNull())
                   .filter(func.col('name') != "")
                   .withColumn('name', udf_unidecode(func.col('name')))
                    .withColumn('name', func.lower(func.col('name')))
                 # .na.drop(subset = 'name')

                         .select('inst_id','name')

                         )
fr_oa_normalized_names.cache()
fr_oa_normalized_names.show()

+-----------+--------------------+
|    inst_id|                name|
+-----------+--------------------+
| I193033237|institut superieu...|
|I4210085936|            labcatal|
|I4210090216|     laboratoire mer|
|I4210096859|centre nouvelle a...|
|I4210101037|laboratoire physi...|
|I4210106601|virtual open systems|
|I4210111182|      impeto medical|
|I4210131541|acies consulting ...|
|I4210134453|     fondation arcad|
|I4210134684|ferring pharmaceu...|
|I4210141217|            vizion'r|
|I4210141565|institut chimie o...|
|I4210152014|hopital georges c...|
|I4387153895|maison sciences h...|
|I2801819261|association futur...|
|I4210091226|ecole superieure ...|
|I4210091847|           cermavein|
|I4210092913|  tethys instruments|
|I4210103254|centre microbiolo...|
|I4210109693|association reche...|
+-----------+--------------------+
only showing top 20 rows



In [18]:
fr_oa_normalized_names.filter(func.col('name')=="evosens").show()

+-----------+-------+
|    inst_id|   name|
+-----------+-------+
|I4210147280|evosens|
+-----------+-------+



In [47]:
budget_normalized_names = (budget
                           .withColumn('stopwords_fr',func.array([func.lit(x) for x in stopwords_dict['French']]))
                           .withColumn('name', udf_rm_stopwords(func.lower(func.col('etablissement')), func.col('stopwords_fr')))

                           .withColumn('name', func.regexp_replace(func.col('name'),   r"""[!\"#$%&'()*+,\-.\/:;<=>?@\[\\\]^_`{|}~]""", ' '))
                           .withColumn('name', udf_rm_stopwords(func.lower(func.col('name')), func.col('stopwords_fr')))
                           .withColumn('name', udf_unidecode(func.col('name')))
                           .withColumn('name', func.regexp_replace(func.col('name'),  r"\s{2,}", ' '))
                           .withColumn('name', func.regexp_replace(func.col('name'), r"\([A-z\s]*\)", ''))
                           .withColumn('name', func.regexp_replace(func.col('name'), r"\s$", ''))
                           .filter(func.col('name').isNotNull())
                           .filter(func.col('name') != "")

                           .select('etablissement','etablissement_id_paysage','name')
                         )
budget_normalized_names.cache()
budget_normalized_names.show()

+--------------------+------------------------+--------------------+
|       etablissement|etablissement_id_paysage|                name|
+--------------------+------------------------+--------------------+
|Observatoire de l...|                   8JlNN|observatoire cote...|
|Observatoire de l...|                   8JlNN|observatoire cote...|
|Observatoire de l...|                   8JlNN|observatoire cote...|
|Université NICE -...|                   7CYWd|universite nice s...|
|COMUE université ...|                   RDy6Q|comue universite ...|
|COMUE université ...|                   RDy6Q|comue universite ...|
|EPE université Cô...|                   s3t8T|epe universite co...|
|EPE université Cô...|                   s3t8T|epe universite co...|
|Université de tec...|                   8tVLr|universite techno...|
|Université de tec...|                   8tVLr|universite techno...|
|Institut d'études...|                   G8QBG|institut etudes p...|
|Institut d'études...|            

In [49]:
merge_inst_by_names = (fr_oa_normalized_names
                       .filter(func.col('name')!='')

                       .join(budget_normalized_names.filter(func.col('name')!='')
                             ,on = ['name'], how = 'inner')
                        .distinct()
                        .withColumn('several_id', 
                         ((func.size(func.collect_set('inst_id').over(Window.partitionBy('name')))>1) |
                         (func.size(func.collect_set('etablissement_id_paysage').over(Window.partitionBy('name')))>1)).cast('int'))

                  )


In [50]:
merge_inst_by_names.persist()
merge_inst_by_names.select('etablissement_id_paysage','inst_id').distinct().count()


63

In [65]:
merge_inst_by_names.filter(func.col('name').contains('bordeaux')).show(truncate = False)

+-------------------------------+-----------+----------------------------------+------------------------+----------+
|name                           |inst_id    |etablissement                     |etablissement_id_paysage|several_id|
+-------------------------------+-----------+----------------------------------+------------------------+----------+
|universite bordeaux            |I15057530  |Université BORDEAUX               |90I54                   |0         |
|institut polytechnique bordeaux|I4210160189|Institut polytechnique de Bordeaux|S88MV                   |0         |
+-------------------------------+-----------+----------------------------------+------------------------+----------+



In [53]:
unmatched = (budget_normalized_names
             .join(merge_inst_by_names.select('etablissement_id_paysage'), on = 'etablissement_id_paysage', how = 'leftanti')
            )
unmatched.show(truncate = False)

+------------------------+--------------------------------------------+---------------------------------------+
|etablissement_id_paysage|etablissement                               |name                                   |
+------------------------+--------------------------------------------+---------------------------------------+
|8JlNN                   |Observatoire de la Côte d'Azur              |observatoire cote azur                 |
|8JlNN                   |Observatoire de la Côte d'Azur              |observatoire cote azur                 |
|8JlNN                   |Observatoire de la Côte d'Azur              |observatoire cote azur                 |
|7CYWd                   |Université NICE - SOPHIA-ANTIPOLIS          |universite nice sophia antipolis       |
|RDy6Q                   |COMUE université Côte d’Azur                |comue universite cote d'azur           |
|RDy6Q                   |COMUE université Côte d’Azur                |comue universite cote d'azur     

In [55]:
unmatched.distinct().toPandas().to_csv( scanR_path + '\\budget_inst_to_match.csv')

In [233]:
handmatched = spark.read.csv('file:///' + scanR_path + '\\budget_inst_handmatched.csv', header = True)

In [234]:
all_matches = (merge_inst_by_names
               .select('etablissement_id_paysage', 'inst_id').distinct()
               .unionAll(handmatched
               .select('etablissement_id_paysage', 'inst_id').distinct()
                        )
              )

In [236]:
all_matches.count()

198

In [254]:
all_matches.filter(func.col('inst_id')=="I4210137101").show()

+------------------------+-------+
|etablissement_id_paysage|inst_id|
+------------------------+-------+
+------------------------+-------+



In [245]:
budget.select('exercice').distinct().collect()

[Row(exercice='2016'),
 Row(exercice='2012'),
 Row(exercice='2020'),
 Row(exercice='2019'),
 Row(exercice='2017'),
 Row(exercice='2014'),
 Row(exercice='2013'),
 Row(exercice='2018'),
 Row(exercice='2011'),
 Row(exercice='2022'),
 Row(exercice='2023'),
 Row(exercice='2021'),
 Row(exercice='2024'),
 Row(exercice='2015'),
 Row(exercice='2010')]

In [259]:
inst_budget = (inst
               .withColumn('parent_id', func.explode_outer(func.col('parent.parent_id')))
               .join(all_matches.withColumnRenamed('inst_id','parent_id'), on = 'parent_id', how = 'left')
               .join(budget.withColumnRenamed('exercice', 'year'), on = 'etablissement_id_paysage', how = 'left')
               .groupBy('inst_id','year')
               .agg(*[func.sum(func.col(col)).alias(col) for col in budget.columns
                      if col not in ['etablissement','etablissement_id_paysage','exercice','academie','code_de_l_academie','code_de_la_commune','code_de_la_region',
                                     'region','source','commune','uai_identifiant','vague']
                     ]
                   )
              )

In [260]:
inst_budget.show()

+-----------+----+------------------------------+---------------------------------+----------------------------+-------------------------+------------------+----------------------------+----------------------------------+-------------------------+--------------------------+--------------------------------------+--------------------+------------------------------------------+------------------------------------------+----------------+--------------------------------------+---------------------------------------------+--------------------+--------------------------------+---------------------------------------------------+-----------------------------+------------------------------------------+---------------------------------------+------+------------------+-------------------------------+----------------------------------------+----------------------+-------------------------------+----------------+------------------------+----------------------------+----------------------------------

In [261]:
inst_budget.toPandas().to_csv( scanR_path + '\\budget_inst_matched.csv')

In [262]:
spark.catalog.clearCache()
spark.stop()